# 🎯 Huấn luyện mô hình Random Forest (Final Optimized)
Dựa trên phân tích thực tế, mô hình Decision Tree bị Overfitting. Tệp này chứa quy trình chuẩn hóa dữ liệu bằng `RobustScaler` và huấn luyện mô hình `Random Forest Classifier` để đạt hiệu suất tối ưu.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 1. Load và Tiền xử lý dữ liệu

In [ ]:
print("1️⃣ Loading dataset...")
columns_to_keep = [
    'Seq', 'Mean', 'sTos', 'sTtl', 'dTtl', 'sHops', 'TotBytes', 
    'SrcBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'SrcWin', 
    'TcpRtt', 'AckDat', 'Label', ' e        ', ' e d      ', 
    'icmp', 'tcp', 'CON', 'FIN', 'INT', 'REQ', 'RST', 'Status'
]

df = pd.read_csv(r'D:\Study\DH\IoT in 5G\dataset\dataset5g-nidd\Encoded\Encoded.csv', usecols=columns_to_keep)

# Xử lý missing values
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].median(), inplace=True)
    if df['Label'].isnull().any():
        df.dropna(subset=['Label'], inplace=True)

# Xử lý infinite values
for col in df.select_dtypes(include=[np.number]).columns:
    if np.isinf(df[col]).any():
        df[col].replace([np.inf, -np.inf], df[col].median(), inplace=True)

print(f"✓ Loaded {len(df):,} samples")

## 2. Feature Engineering
Trích xuất 3 tính năng phái sinh: `BytesRatio`, `PktSizeRatio`, và `TTLDiff`.

In [ ]:
# Move Label to last temporarily
column_to_move = df.pop('Label')

# Feature engineering
df['BytesRatio'] = df['SrcBytes'] / (df['TotBytes'] + 1)
df['PktSizeRatio'] = df['sMeanPktSz'] / (df['dMeanPktSz'] + 1)
df['TTLDiff'] = np.abs(df['sTtl'] - df['dTtl'])

# Put Label back
df['Label'] = column_to_move

feature_names = df.columns[:-1].tolist()
print(f"✓ Engineered Features. Total features: {len(feature_names)}")

## 3. Scale Dữ liệu và Tách tập Train/Test

In [ ]:
X = df.iloc[:, :-1].values
y = df['Label'].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Data prepared: Train={len(X_train):,}, Val={len(X_val):,}, Test={len(X_test):,}")

## 4. Huấn luyện Random Forest

In [ ]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
print("✓ Training complete!")

## 5. Đánh giá Mô hình trên tập Test

In [ ]:
y_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 6. Lưu Mô hình

In [ ]:
from datetime import datetime
import os
os.makedirs('../model', exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f'../model/random_forest_model_OPTIMIZED_{timestamp}.pkl'
scaler_path = f'../model/scaler_OPTIMIZED_{timestamp}.pkl'
features_path = f'../model/feature_names_OPTIMIZED_{timestamp}.pkl'

joblib.dump(rf_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_names, features_path)

print(f"💾 Best model saved:")
print(f"  Model:    {model_path}")
print(f"  Scaler:   {scaler_path}")
print(f"  Features: {features_path}")